In [ ]:
# --- INSTALLATION (Run this first if needed) ---
# !pip install openai beautifulsoup4 requests python-dotenv

In [1]:
import os
import json
import requests_oauthlib
from bs4 import BeautifulStoneSoup
from openai import OpenAI
from IPython.display import Markdown, display

In [11]:
# --- PILLAR 4: THE BRAIN SETUP ---
# Replace with your actual OpenRouter Key
# OPENROUTER_API_KEY = "your_key_here"
OPENROUTER_API_KEY = "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"


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key= OPENROUTER_API_KEY
)

# You can use "google/gemini-2.0-flash-lite" for speed/low cost
MODAL="google/gemini-2.0-flash-lite" 

# --- PILLAR 1 & 3: THE SCRAPER & AGGREGATOR ---
def fetch_website_links(url):
    """Vacuum cleaner for links"""
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        links = [a.get('href') for a in soup.find_all('a', href=True)]
        from urllib.parse import urljoin
        return [urljoin(url, l) for l in links]
    except Exception as e:
        return []

def fetch_website_contents(url):
    """Vacuum cleaner for text"""
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        for script_or_style in soup(["script", "style"]):
            script_or_style.extract()
        return soup.get_text(separator=' ', strip=True)
    except Exception as e:
        return ""
# --- PILLAR 2: THE SMART FILTER ---
link_system_prompt = """
You are a Link Filter. Look at these links and pick the ones most relevant for a company brochure 
(About, Careers, Products). Respond ONLY in JSON:
{"links": [{"type": "page type", "url": "full_url"}]}
"""

def select_relevant_links(url):
    links = fetch_website_links(url)
    user_prompt = f"Links from {url}:\n" + "\n".join(links[:50])
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

# --- PILLAR 4: THE LOGIC & PERSONA ---
brochure_system_prompt = """
You are a Senior Business Analyst. Create a professional company brochure in Markdown.
Include details on Culture, Customers, and Careers. Do not use code blocks.
"""

def create_brochure(company_name, url):
    print(f"🏗️ Building brochure for {company_name}...")
    
    # Pillar 3: Aggregating data
    landing_text = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    
    all_context = f"Main Page: {landing_text}\n"
    for link in relevant_links['links']:
        all_context += f"\nSub-page ({link['type']}): " + fetch_website_contents(link['url'])
    
    # Pillar 4: Processing the Brain
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": f"Data: {all_context[:10000]}"} # Token protection
        ]
    )
    display(Markdown(response.choices[0].message.content))

# --- THE TEST DRIVE ---
create_brochure("HuggingFace", "https://huggingface.co")


🏗️ Building brochure for HuggingFace...


NameError: name 'MODEL' is not defined

In [ ]:
# --- INSTALLATION (Run this first if needed) ---
# !pip install openai beautifulsoup4 requests python-dotenv

import os
import json
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from IPython.display import Markdown, display

# --- PILLAR 4: THE BRAIN SETUP ---
# Replace with your actual OpenRouter Key
OPENROUTER_API_KEY = "your_key_here" 


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# You can use "google/gemini-2.0-flash-lite" for speed/low cost
MODEL = "google/gemini-2.0-flash-001"

# --- PILLAR 1 & 3: THE SCRAPER & AGGREGATOR ---
def fetch_website_links(url):
    """Vacuum cleaner for links"""
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        links = [a.get('href') for a in soup.find_all('a', href=True)]
        from urllib.parse import urljoin
        return [urljoin(url, l) for l in links]
    except Exception as e:
        return []

def fetch_website_contents(url):
    """Vacuum cleaner for text"""
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        for script_or_style in soup(["script", "style"]):
            script_or_style.extract()
        return soup.get_text(separator=' ', strip=True)
    except Exception as e:
        return ""

# --- PILLAR 2: THE SMART FILTER ---
link_system_prompt = """
You are a Link Filter. Look at these links and pick the ones most relevant for a company brochure 
(About, Careers, Products). Respond ONLY in JSON:
{"links": [{"type": "page type", "url": "full_url"}]}
"""

def select_relevant_links(url):
    links = fetch_website_links(url)
    user_prompt = f"Links from {url}:\n" + "\n".join(links[:50])
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

# --- PILLAR 4: THE LOGIC & PERSONA ---
brochure_system_prompt = """
You are a Senior Business Analyst. Create a professional company brochure in Markdown.
Include details on Culture, Customers, and Careers. Do not use code blocks.
"""

def create_brochure(company_name, url):
    print(f" Building brochure for {company_name}...")
    
    # Pillar 3: Aggregating data
    landing_text = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    
    all_context = f"Main Page: {landing_text}\n"
    for link in relevant_links['links']:
        all_context += f"\nSub-page ({link['type']}): " + fetch_website_contents(link['url'])
    
    # Pillar 4: Processing the Brain
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": f"Data: {all_context[:10000]}"} # Token protection
        ]
    )
    display(Markdown(response.choices[0].message.content))

# --- THE TEST DRIVE ---
create_brochure("HuggingFace", "https://huggingface.co")